# Treja POC — RTMDet-Small → ONNX (raw, no NMS)

Run this whole notebook top to bottom (`Runtime > Run all`). At the end:
1. You'll see a test image with detected objects drawn on it, directly in this notebook — that's our sanity check that the export is correct.
2. A file called `rtmdet_small_raw.onnx` will auto-download to your computer.

That file is what goes into the phone-testable webpage next. No local install needed — everything runs on Google's servers.

## 1. Install dependencies (Colab has prebuilt wheels for all of this — should be fast, no compiler needed)

In [ ]:
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q "mmcv==2.1.0"
!pip install -q mmdet==3.3.0
!pip install -q onnx onnxruntime opencv-python-headless matplotlib

## 2. Download the official RTMDet-Small COCO checkpoint (pretrained by OpenMMLab, Apache-2.0)

In [ ]:
!mkdir -p /content/ckpt
!mim download mmdet --config rtmdet-small_8xb32-300e_coco --dest /content/ckpt
import glob
config_file = glob.glob('/content/ckpt/*.py')[0]
checkpoint_file = glob.glob('/content/ckpt/*.pth')[0]
print('config:', config_file)
print('checkpoint:', checkpoint_file)

## 3. Load the model

In [ ]:
import torch
from mmdet.apis import init_detector

model = init_detector(config_file, checkpoint_file, device='cpu')
model.eval()
print('num_classes:', model.bbox_head.num_classes)
print('strides:', model.bbox_head.prior_generator.strides)
print(model.data_preprocessor.mean, model.data_preprocessor.std, model.data_preprocessor.bgr_to_rgb)

## 4. Wrapper module: raw decoded boxes + scores, NO NMS

We stop right before NMS because ONNX/browser export can't include RTMDet's NMS step (it needs a custom native operator that doesn't exist in-browser). We use mmdetection's own real `distance2bbox` decode function and real prior-point generator — not a reimplementation — so this matches the library's actual math. NMS itself (simple, standard) will be done in JavaScript on the phone.

In [ ]:
from mmdet.structures.bbox import distance2bbox

class RTMDetRawExport(torch.nn.Module):
    def __init__(self, det_model):
        super().__init__()
        self.backbone = det_model.backbone
        self.neck = det_model.neck
        self.bbox_head = det_model.bbox_head

    def forward(self, img):
        feats = self.neck(self.backbone(img))
        cls_scores, bbox_preds = self.bbox_head(feats)

        featmap_sizes = [c.shape[-2:] for c in cls_scores]
        mlvl_priors = self.bbox_head.prior_generator.grid_priors(
            featmap_sizes, dtype=cls_scores[0].dtype, device=cls_scores[0].device)

        flatten_cls, flatten_bbox = [], []
        for cls_score, bbox_pred in zip(cls_scores, bbox_preds):
            b, c, h, w = cls_score.shape
            flatten_cls.append(cls_score.permute(0, 2, 3, 1).reshape(b, h * w, c))
            bb, bc, bh, bw = bbox_pred.shape
            flatten_bbox.append(bbox_pred.permute(0, 2, 3, 1).reshape(bb, bh * bw, bc))

        cls_scores_cat = torch.cat(flatten_cls, dim=1)   # (1, N, 80)
        bbox_preds_cat = torch.cat(flatten_bbox, dim=1)  # (1, N, 4)
        priors_cat = torch.cat(mlvl_priors, dim=0)       # (N, 2) -> [x, y] centers

        bboxes = distance2bbox(priors_cat[None, :, :2], bbox_preds_cat)  # (1, N, 4) xyxy, absolute pixels
        scores = cls_scores_cat.sigmoid()                                 # (1, N, 80)
        return bboxes, scores

export_model = RTMDetRawExport(model)
export_model.eval()

dummy = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    b, s = export_model(dummy)
print('boxes shape:', b.shape, '| scores shape:', s.shape)

## 5. Sanity check — run on a real image, compare against mmdetection's OWN official inference

This is the critical check: we run our raw export (then apply simple score-threshold + NMS ourselves) and compare it side-by-side against the model's official, trusted `predict_by_feat` output on the exact same image. If the boxes match, our export math is correct.

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/open-mmlab/mmdetection/main/demo/demo.jpg',
    '/content/test.jpg')

import cv2, numpy as np
img_bgr = cv2.imread('/content/test.jpg')
orig_h, orig_w = img_bgr.shape[:2]

# Letterbox resize to 640x640 (keep_ratio=True, pad_val=114), matching RTMDet's own test_pipeline
scale = min(640 / orig_h, 640 / orig_w)
new_h, new_w = int(round(orig_h * scale)), int(round(orig_w * scale))
resized = cv2.resize(img_bgr, (new_w, new_h))
canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
canvas[0:new_h, 0:new_w] = resized

mean = np.array(model.data_preprocessor.mean).reshape(1, 1, 3)
std = np.array(model.data_preprocessor.std).reshape(1, 1, 3)
normed = (canvas.astype(np.float32) - mean) / std   # BGR order, matches bgr_to_rgb=False
tensor_in = torch.from_numpy(normed.transpose(2, 0, 1)[None]).float()

with torch.no_grad():
    raw_boxes, raw_scores = export_model(tensor_in)

# Our own simple score-threshold + per-class NMS (mirrors what JS will do)
from torchvision.ops import nms as tv_nms
score_thr = 0.3
raw_boxes0, raw_scores0 = raw_boxes[0], raw_scores[0]
max_scores, labels = raw_scores0.max(dim=1)
keep = max_scores > score_thr
boxes_k, scores_k, labels_k = raw_boxes0[keep], max_scores[keep], labels[keep]
final_boxes, final_scores, final_labels = [], [], []
for cls_id in labels_k.unique():
    cls_mask = labels_k == cls_id
    idx = tv_nms(boxes_k[cls_mask], scores_k[cls_mask], iou_threshold=0.5)
    final_boxes.append(boxes_k[cls_mask][idx])
    final_scores.append(scores_k[cls_mask][idx])
    final_labels.append(torch.full((len(idx),), cls_id))
our_boxes = torch.cat(final_boxes) if final_boxes else torch.empty(0, 4)
our_scores = torch.cat(final_scores) if final_scores else torch.empty(0)
our_labels = torch.cat(final_labels) if final_labels else torch.empty(0)
print(f'Our raw-export + manual NMS: {len(our_boxes)} detections')

# Official mmdetection inference on the same preprocessed tensor, for comparison
with torch.no_grad():
    feats = model.extract_feat(tensor_in)
    cls_scores, bbox_preds = model.bbox_head(feats)
    from mmdet.structures import DetDataSample
    from mmengine.structures import InstanceData
    meta = {'img_shape': (640, 640), 'scale_factor': (1.0, 1.0)}
    official = model.bbox_head.predict_by_feat(
        cls_scores, bbox_preds, batch_img_metas=[meta], cfg=model.test_cfg,
        rescale=False, with_nms=True)[0]
print(f'Official mmdetection inference: {len(official.bboxes)} detections')

In [ ]:
import matplotlib.pyplot as plt
from mmdet.datasets.coco import CocoDataset
COCO_CLASSES = CocoDataset.METAINFO['classes']

vis = cv2.cvtColor(canvas.copy(), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for box, lbl, sc in zip(our_boxes, our_labels, our_scores):
    x1, y1, x2, y2 = box.int().tolist()
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
    cv2.putText(vis, f'{COCO_CLASSES[int(lbl)]} {sc:.2f}', (x1, max(y1 - 5, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
axes[0].imshow(vis); axes[0].set_title(f'OUR raw ONNX-path export ({len(our_boxes)} objects)'); axes[0].axis('off')

vis2 = cv2.cvtColor(canvas.copy(), cv2.COLOR_BGR2RGB)
for box, lbl, sc in zip(official.bboxes, official.labels, official.scores):
    x1, y1, x2, y2 = box.int().tolist()
    cv2.rectangle(vis2, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(vis2, f'{COCO_CLASSES[int(lbl)]} {sc:.2f}', (x1, max(y1 - 5, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
axes[1].imshow(vis2); axes[1].set_title(f'OFFICIAL mmdetection inference ({len(official.bboxes)} objects)'); axes[1].axis('off')
plt.savefig('/content/comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print()
print('=' * 70)
print('LOOK AT THE TWO IMAGES ABOVE.')
print('If the red boxes (ours) closely match the green boxes (official),')
print('the export is verified correct. If they look very different,')
print('STOP and send this notebook output back — do not use the ONNX file.')
print('=' * 70)

## 6. Export to ONNX (only run/trust this after confirming step 5 looks correct)

In [ ]:
torch.onnx.export(
    export_model,
    dummy,
    '/content/rtmdet_small_raw.onnx',
    input_names=['image'],
    output_names=['boxes', 'scores'],
    opset_version=17,
    do_constant_folding=True,
)

import onnx
onnx_model = onnx.load('/content/rtmdet_small_raw.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX model is valid.')
print('Inputs:', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in onnx_model.graph.input])
print('Outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in onnx_model.graph.output])

import os
print(f"File size: {os.path.getsize('/content/rtmdet_small_raw.onnx') / 1e6:.1f} MB")

## 7. Re-verify the actual exported ONNX file (not just the PyTorch wrapper) with ONNX Runtime, then download

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession('/content/rtmdet_small_raw.onnx')
ort_boxes, ort_scores = sess.run(None, {'image': tensor_in.numpy()})
print('ONNX Runtime output shapes:', ort_boxes.shape, ort_scores.shape)
diff = np.abs(ort_boxes - raw_boxes.numpy()).max()
print(f'Max difference vs PyTorch (should be ~0): {diff:.6f}')

from google.colab import files
files.download('/content/rtmdet_small_raw.onnx')
files.download('/content/comparison.png')
print('Downloads triggered — check your browser downloads.')